In [0]:
# ---------------------------------------------------------------------------
# GOLD: daily per-city aggregates, ready for BI / reporting consumption.
# ---------------------------------------------------------------------------

dbutils.widgets.text("silver_table", "silver.weather_hourly", "Silver Delta table")
dbutils.widgets.text("gold_table", "gold.weather_daily_summary", "Gold Delta table")

silver_table = dbutils.widgets.get("silver_table")
gold_table = dbutils.widgets.get("gold_table")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

df_silver = spark.table(silver_table)

df_gold = (
    df_silver
    .withColumn("observation_date", F.to_date("observation_ts"))
    .groupBy("city", "observation_date")
    .agg(
        F.avg("temperature_c").alias("avg_temperature_c"),
        F.min("temperature_c").alias("min_temperature_c"),
        F.max("temperature_c").alias("max_temperature_c"),
    )
    .withColumn("_processed_ts", F.current_timestamp())
)

In [0]:
if spark.catalog.tableExists(gold_table):
    target = DeltaTable.forName(spark, gold_table)
    (
        target.alias("t")
        .merge(df_gold.alias("s"), "t.city = s.city AND t.observation_date = s.observation_date")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_gold.write.format("delta").saveAsTable(gold_table)

row_count = df_gold.count()
print(f"Gold aggregate complete: {row_count} daily rows merged into {gold_table}")

In [0]:
dbutils.notebook.exit(str(row_count))